# Tables

The Hugoniot fit data and replication records are available in directory <a href="./_static/model-fits/">model-fits</a>.


In [1]:
%%capture 
## This cell loads the modules, data, and EOS parameters
## capture magic function hides the output; comment out to see diagnostic info during data load
## Run executable notebook for output

%run -i scripts/set_paths
%run -i scripts/load_modules
%run -i scripts/load_silica_data
%run -i scripts/define_mc
%run -i scripts/define_eos # this notebook requires variables from this script
%run -i scripts/intersection.py # intersection function


In [2]:
def curve2num(curvetype):
    if curvetype == 'linear':
        return 1
    if curvetype == 'quadratic':
        return 2
    if curvetype == 'cubic':
        return 3
    if curvetype == 'fourthpoly':
        return 4
def myform(x):
    s = print('%12.6e' % x )
    return str(s)

def format_matrix(matrix, environment="bmatrix", formatter=lambda s: '%12.6e' % s):
    """Format a matrix using LaTeX syntax"""

    if not isinstance(matrix, np.ndarray):
        try:
            matrix = np.array(matrix)
        except Exception:
            raise TypeError("Could not convert to Numpy array")

    if len(shape := matrix.shape) == 1:
        matrix = matrix.reshape(1, shape[0])
    elif len(shape) > 2:
        raise ValueError("Array must be 2 dimensional")

    body_lines = [" & ".join(map(formatter, row)) for row in matrix]

    body = "\\\\\n".join(body_lines)
    return f"""\\begin{{{environment}}}
{body}
\\end{{{environment}}}"""

def tomatrix(fit,var,label):
    s1='% '+fit.filename+' \n'
    s2='\\label{'+label+'} \n'
    s3 = '\\Sigma'+var+' &=& '+format_matrix2(fit.cov) + ' \\\\'
    print(s1+s2+s3)

def fnum(a):
    s = ''
    if abs(a) < 100000 and abs(a) > 1.e-5:
        s += '%12.6f' % a
    else:
        s += '%12.6e' % a
    return s
def fnum2(a):
    return '%12.6e' % a

def xfrexp10(x):
    exp = int(np.floor(np.log10(abs(x))))
    if exp != 0:
        s = str(x / 10**exp)[0:8]+' \\times 10^{'+str(exp)+'}'
    else:
        s = str(x / 10**exp)[0:8]        
    return s
def frexp10(x):
    #if x != 0:
    #    exp = int(np.floor(np.log10(abs(x))))
    #if exp != 0:
    s = '\\textrm{'+fnum2(x)+'}'
    #else:
    #    s = str(x)[0:8]        
    return s    
#def format_matrix2(matrix, environment="bmatrix", \
#                   formatter=lambda s: '%12.6f' % s):
def format_matrix2(matrix, environment="bmatrix", \
                   formatter=frexp10):
        
    """Format a matrix using LaTeX syntax"""

    if not isinstance(matrix, np.ndarray):
        try:
            matrix = np.array(matrix)
        except Exception:
            raise TypeError("Could not convert to Numpy array")

    if len(shape := matrix.shape) == 1:
        matrix = matrix.reshape(1, shape[0])
    elif len(shape) > 2:
        raise ValueError("Array must be 2 dimensional")

    body_lines = [" & ".join(map(formatter, row)) for row in matrix]
#    body_lines = [" & ".join(str(row)) for row in matrix]

    body = "\\\\\n".join(body_lines)
    return f"""\\begin{{{environment}}}
{body}
\\end{{{environment}}}"""

def formata(a):
    s='['
    for i in range(len(a)):
        if max(abs(a)) < 100000 and min(abs(a)) > 1.e-5:
            s += '%12.6f' % a[i]
        else:
            s += '%12.6e' % a[i]
        if i < len(a)-1:
            s += ', '
        else:
            s += ']'
    return s

def formatstr(val,err,n):
    #print(val,err,n)
    #print(str(int(np.round(val*np.pow(10,n))))[-1:])
    if str(int(np.round(val*np.pow(10,n))))[-1:] == '0':
        sss=(str(np.round(val*np.pow(10,n))/np.pow(10,n))+'0 ± '+str(np.round(err*np.pow(10,n))/np.pow(10,n)))
    else:
        sss=(str(np.round(val*np.pow(10,n))/np.pow(10,n))+' ± '+str(np.round(err*np.pow(10,n))/np.pow(10,n)))
    if n == 0:
        sss=(str(int(np.round(val*np.pow(10,n))))+' ± '+str(int(np.round(err*np.pow(10,n)))))
        
    return sss


In [3]:
print(frexp10(5.2345111116e10))
print(frexp10(5.23456e-4))
print(frexp10(5.23458886))
print(fnum(5.23456e10))
print(fnum(5.23456e4))

tomatrix(qtz_fit_stv0,'_{U_s}','eq:qtz_stv_cov')

format_matrix2(qtz_fit_stv0.cov)

\textrm{5.234511e+10}
\textrm{5.234560e-04}
\textrm{5.234589e+00}
5.234560e+10
52345.600000
% qtz_fit_stv_linear 
\label{eq:qtz_stv_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{4.325045e-02} & \textrm{-1.350526e-02}\\
\textrm{-1.350526e-02} & \textrm{4.256561e-03}
\end{bmatrix} \\


'\\begin{bmatrix}\n\\textrm{4.325045e-02} & \\textrm{-1.350526e-02}\\\\\n\\textrm{-1.350526e-02} & \\textrm{4.256561e-03}\n\\end{bmatrix}'

## Table 1: Summary of gas gun shock data for quartz and fused silica.


In [4]:
# load previously saved picked data
#with open('./imp-match-UCD/silicadata-N1000.pkl','rb') as f:
with open(silicadatadir+'Amodeo-analysis/silicadata-N1000.pkl','rb') as f:
    silicadata_list=pickle.load(f)
print(len(silicadata_list))

19


In [5]:
# DO NOT LOAD OTHER DATA HERE - conflicts between data structures?
qtz_list = []
fs_list = []
qtz_p_list = []
fs_p_list = []


for i in range(len(silicadata_list)):
    print(i,silicadata_list[i].material[0],silicadata_list[i].ref[0],silicadata_list[i].P[0],silicadata_list[i].P_std[0],silicadata_list[i].T[0],silicadata_list[i].T_rel[0],silicadata_list[i].streak_flag[0],silicadata_list[i].T_source[0],silicadata_list[i].T_rel_source[0])
    if (silicadata_list[i].material[0] == 'Fused Silica'):
        fs_list.append(silicadata_list[i])
        fs_p_list.append(silicadata_list[i].P[0])
    else:
        qtz_list.append(silicadata_list[i])
        qtz_p_list.append(silicadata_list[i].P[0])

print(np.argsort(np.asarray(fs_p_list)))

0 Z-cut Alpha-quartz shot14 87.5445084178645 1.0961601914459942 4661.0 -999.0 1 streak nan
1 Fused Silica shot20 94.18468915427654 2.7420223429194777 5027.3122 4995.945309 0 IR4 IR4
2 Fused Silica shot21 101.82350613298814 2.063588557962232 6703.340637 4455.373225 0 IR4 IR3
3 Fused Silica shot22 99.66594921671764 6.376532600175746 4573.230527 4278.540428 1 IR4 IR4
4 Fused Silica shot23 82.97301283075393 1.6254758879955082 4200.1755 4172.764445 0 IR4 IR4
5 Fused Silica shot25 84.26050204345043 4.958932739593444 4483.578691 3496.421395 0 IR2 IR2
6 Z-cut Alpha-quartz shot26 124.25822447224618 4.3949106616268585 5329.4 4096.2 1 streak streak
7 Z-cut Alpha-quartz shot27 65.06362476276111 1.6127713359043017 3412.766828 2616.169 1 streak streak
8 Z-cut Alpha-quartz shot28 68.09205635462091 3.5674273534746135 4142.510301 3160.742536 0 IR4 IR4
9 Fused Silica shot31 83.12356134684009 4.573254027730589 4611.0908 4558.900819 0 IR4 IR4
10 Z-cut Alpha-quartz shot33 120.45526920071242 0.3213371489271

In [6]:
dict(qtz_list[0])

{'material': ['Z-cut Alpha-quartz'],
 'rho0': [np.float64(2.65)],
 'rho0_std': [np.float64(0.005)],
 'uflyer': [np.float64(5.467)],
 'uflyer_std': [np.float64(0.062)],
 'uflyer_units': ['km/s'],
 'up': [np.float64(3.904654148326888)],
 'up_std': [np.float64(0.0480369189869452)],
 'up_units': ['km/s'],
 'us': [np.float64(8.46)],
 'us_std': [np.float64(0.015)],
 'us_units': ['km/s'],
 'P': [np.float64(87.5445084178645)],
 'P_std': [np.float64(1.0961601914459942)],
 'P_units': ['GPa'],
 'rho': [np.float64(4.922265298421781)],
 'rho_std': [np.float64(0.0538645615366176)],
 'rho_units': ['g/cm3'],
 'T': [np.float64(4661.0)],
 'T_std': [np.float64(172.0)],
 'T_source': ['streak'],
 'T_rel': [np.float64(-999.0)],
 'T_rel_std': [np.float64(0.0)],
 'T_rel_source': [nan],
 'ref': ['shot14'],
 'calcinfo': ['SCCM paper; NIR fielded not clear if calibratable'],
 'streak_flag': [np.int64(1)],
 'wiggle_flag': [np.int64(0)]}

In [7]:
for i in range(len(qtz_list)):
    print(qtz_list[i].ref)

['shot14']
['shot26']
['shot27']
['shot28']
['shot33']
['shot44']
['shot45']
['shot47']
['shot48']


In [8]:
IM_list = ['shot22','shot28','shot44','shot50']
decay_list = ['shot50','shot60']
cu_list = ['shot50']
twowave_list = ['shot44','shot27','shot28']

isort = np.argsort(np.asarray(qtz_p_list))
print(isort)
for i in isort:
    print(i,qtz_p_list[i],qtz_list[i].ref[0],qtz_list[i].rho0[0],qtz_list[i].rho0_std[0],qtz_list[i].uflyer[0],qtz_list[i].uflyer_std[0],qtz_list[i].P[0],qtz_list[i].P_std[0],qtz_list[i].us[0],qtz_list[i].us_std[0],qtz_list[i].up[0],qtz_list[i].up_std[0],qtz_list[i].rho[0],qtz_list[i].rho_std[0],qtz_list[i].T[0],qtz_list[i].T_std[0])

print('latex quartz')
isort = np.argsort(np.asarray(qtz_p_list))
print(isort)
for i in isort:
    sss = qtz_list[i].ref[0]
    slabel = sss[4::]
    if sss in IM_list:
        slabel += '*'
    if sss in twowave_list:
        slabel += '§'
    #print(slabel,sss)
    print(formatstr(qtz_list[i].rho0[0],qtz_list[i].rho0_std[0],3),' & ', formatstr(qtz_list[i].uflyer[0],qtz_list[i].uflyer_std[0],3),' & ',formatstr(qtz_list[i].P[0],qtz_list[i].P_std[0],1),' & ',formatstr(qtz_list[i].us[0],qtz_list[i].us_std[0],3),' & ',formatstr(qtz_list[i].up[0],qtz_list[i].up_std[0],3),' & ',formatstr(qtz_list[i].rho[0],qtz_list[i].rho_std[0],3),' & ',formatstr(qtz_list[i].T[0],qtz_list[i].T_std[0],0),' & ',slabel,' \\\\ % '+sss)

print('latex fused silica')
isort = np.argsort(np.asarray(fs_p_list))
print(isort)
for i in isort:
    sss = fs_list[i].ref[0]
    slabel = sss[4::]
    if sss in IM_list:
        slabel += '*'
    if sss in decay_list:
        slabel += '‡'
    if sss in cu_list:
        slabel += '†'
    #print(slabel,sss)
    print(formatstr(fs_list[i].rho0[0],fs_list[i].rho0_std[0],3),' & ', formatstr(fs_list[i].uflyer[0],fs_list[i].uflyer_std[0],3),' & ',formatstr(fs_list[i].P[0],fs_list[i].P_std[0],1),' & ',formatstr(fs_list[i].us[0],fs_list[i].us_std[0],3),' & ',formatstr(fs_list[i].up[0],fs_list[i].up_std[0],3),' & ',formatstr(fs_list[i].rho[0],fs_list[i].rho_std[0],3),' & ',formatstr(fs_list[i].T[0],fs_list[i].T_std[0],0),' & ',slabel,' \\\\ % '+sss)
print('Shot 50 had copper flyer; the rest were SS304')


[5 2 3 8 0 7 6 4 1]
5 65.05552805802914 shot44 2.641 0.002 4.578 0.003 65.05552805802914 1.4738042665703475 7.401 0.215 3.327533523297992 0.023712952455188663 4.8053830945875315 0.14654461115560857 3480.0 60.0
2 65.06362476276111 shot27 2.641 0.002 4.569 0.101 65.06362476276111 1.6127713359043017 7.43 0.052 3.3149103079771622 0.08016215412408277 4.76945889922086 0.0989248706910648 3412.766828 91.982392
3 68.09205635462091 shot28 2.65 0.002 4.705 0.2 68.09205635462091 3.5674273534746135 7.553 0.23 3.4009971466589524 0.15941846520734648 4.834729271960871 0.2390641445105797 4142.510301 169.457238
8 82.99228665943825 shot48 2.651 0.003 5.278 0.004 82.99228665943825 0.13207397327859763 8.2994 0.0042 3.772212268154362 0.004965606557962595 4.859786442352708 0.007451346814617761 4136.765735 169.151493
0 87.5445084178645 shot14 2.65 0.005 5.467 0.062 87.5445084178645 1.0961601914459942 8.46 0.015 3.904654148326888 0.0480369189869452 4.922265298421781 0.0538645615366176 4661.0 172.0
7 102.461287

## Table 2: Summary of Hugoniot fits.
HPfit_EOS objects are loaded in define_eos.py.<br>
This notebook creates latex formatted entries for Table 5 and Appendix D.


### Fused Silica

Stishovite range:<br>
Onset of stv diffraction at 34 GPa (Tracy et al.)<br>
Transition to CaCl2 depends on choice of phase boundary. Here using GGA from Tsuchiya et al. 2004.<br>


In [9]:
formata(fs_fit_stv0.a)
formata(fs_fitTus_stv0.a)


'[ 7684.978628, -2052.464734,   226.345708]'

In [10]:
#dict(fs_fit_stv0)
#dict(fs_fitTus_stv0)
#dict(fs_fit_liq0)

In [11]:

liqmaxP = np.interp(fs_fit_liq0.xrange[1],fs_hug_liq.up,fs_hug_liq.P)
print(liqmaxP)
print(curve2num(fs_fit_liq0.curvetype))

stvTmaxP = np.interp(fs_fitTus_stv0.xrange[1],fs_hug_stv.us,fs_hug_stv.P)
stvTminP = np.interp(fs_fitTus_stv0.xrange[0],fs_hug_stv.us,fs_hug_stv.P)
print(stvTminP,stvTmaxP)

liqTmaxP = np.interp(fs_fitTus_liq0.xrange[1],fs_hug_liq.us,fs_hug_liq.P)
liqTminP = np.interp(fs_fitTus_liq0.xrange[0],fs_hug_liq.us,fs_hug_liq.P)
print(liqTminP,liqTmaxP)

1099.401202196769
2
36.184304358940125 68.0599019463296
73.12833723380406 627.6633949820268


In [12]:
## Original table
# Region & $P$ (GPa) & Form & $f(x)$ & Coefficients $a_i$ & Fit range (km/s) & $N$ & $\Sigma$ & Notes \\

# FS stv us-up
print(fs_fit_stv0.phase_fit, ' & ',\
      int(np.round(fs_trans_stv_P)),'-',int(np.round(fs_startsol_P)),' & ',\
      curve2num(fs_fit_stv0.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(fs_fit_stv0.a),' & ',\
      np.round(fs_fit_stv0.xrange[0]*100)/100,'$< u_p <$',np.round(100*fs_fit_stv0.xrange[1])/100,' & ',\
      fs_fit_stv0.npts,' & ',\
      'Eq. \\ref{eq:fs_stv_cov}',' & ',\
      ' \\\\')
# FS stv T-us
print(fs_fitTus_stv0.phase_fit, ' & ',\
      int(np.round(fs_trans_stv_P)),'-',int(np.round(fs_startsol_P)),' & ',\
      curve2num(fs_fitTus_stv0.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(fs_fitTus_stv0.a),' & ',\
      np.round(100*fs_fitTus_stv0.xrange[0])/100,'$< U_s <$',np.round(100*fs_fitTus_stv0.xrange[1])/100,' & ',\
      fs_fitTus_stv0.npts,' & ',\
      'Eq. \\ref{eq:fs_stv_T_cov}',' & ',\
      ' \\\\')
# FS liq Us-up
print(fs_fit_liq0.phase_fit, ' & ',\
      int(np.round(fs_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      curve2num(fs_fit_liq0.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(fs_fit_liq0.a),' & ',\
      np.round(100*fs_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*fs_fit_liq0.xrange[1])/100,' & ',\
      fs_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:fs_liq_cov}',' & ',\
      '$\\ddagger$ \\\\')
# FS liq T-Us
print(fs_fit_liq0.phase_fit, ' & ',\
      int(np.round(fs_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      curve2num(fs_fitTus_liq0.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(fs_fitTus_liq0.a),' & ',\
      np.round(100.*fs_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*fs_fitTus_liq0.xrange[1])/100,' & ',\
      fs_fitTus_liq0.npts,' & ',\
      'Eq. \\ref{eq:fs_liq_T_cov}',' & ',\
      '$\\ddagger$ \\\\')

stishovite  &  34 - 72  &  1  &  $U_s(u_p)$ &  [    1.107070,     1.607013]  &  2.76 $< u_p <$ 3.91  &  16  &  Eq. \ref{eq:fs_stv_cov}  &   \\
stishovite  &  34 - 72  &  2  &  $T(U_s)$ &  [ 7684.978628, -2052.464734,   226.345708]  &  5.72 $< U_s <$ 7.62  &  21  &  Eq. \ref{eq:fs_stv_T_cov}  &   \\
liquid  &  74 - 1099  &  2  &  $U_s(u_p)$ &  [    1.230166,     1.681657,    -0.014014]  &  4.76 $< u_p <$ 18.28  &  132  &  Eq. \ref{eq:fs_liq_cov}  &  $\ddagger$ \\
liquid  &  74 - 628  &  3  &  $T(U_s)$ &  [-1418.437322,   526.816293,   -12.799697,     4.979201]  &  7.98 $< U_s <$ 21.25  &  42  &  Eq. \ref{eq:fs_liq_T_cov}  &  $\ddagger$ \\


In [13]:
## New table columns
# Region & $f(x)$ & Form & Coefficients $a_i$ & $f,x$ units & Fit range & $P$ range (GPa) & $N$ & Cov. Matrix$^{\Sigma}$ \\

# FS stv us-up
print(fs_fit_stv0.phase_fit, ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(fs_fit_stv0.curvetype), ' & ',\
      formata(fs_fit_stv0.a),' & ',\
      fs_fit_stv0.x_units,', ',fs_fit_stv0.f_units,' & ',\
      np.round(fs_fit_stv0.xrange[0]*100)/100,'$< u_p <$',np.round(100*fs_fit_stv0.xrange[1])/100,' & ',\
      int(np.round(fs_trans_stv_P)),'-',int(np.round(fs_startsol_P)),' & ',\
      fs_fit_stv0.npts,' & ',\
      'Eq. \\ref{eq:fs_stv_cov}','  ',\
      ' \\\\')
# FS stv T-us
print(fs_fitTus_stv0.phase_fit, ' & ',\
      '$T(U_s)$ & ',\
      curve2num(fs_fitTus_stv0.curvetype), ' & ',\
      formata(fs_fitTus_stv0.a),' & ',\
      fs_fitTus_stv0.x_units,', ',fs_fitTus_stv0.f_units,' & ',\
      np.round(100*fs_fitTus_stv0.xrange[0])/100,'$< U_s <$',np.round(100*fs_fitTus_stv0.xrange[1])/100,' & ',\
      int(np.round(fs_trans_stv_P)),'-',int(np.round(fs_startsol_P)),' & ',\
      fs_fitTus_stv0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:fs_stv_T_cov}','  ',\
      ' \\\\')
# FS liq Us-up
print(fs_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(fs_fit_liq0.curvetype), ' & ',\
      formata(fs_fit_liq0.a),' & ',\
      fs_fit_liq0.x_units,', ',fs_fit_liq0.f_units,' & ',\
      np.round(100*fs_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*fs_fit_liq0.xrange[1])/100,' & ',\
      int(np.round(fs_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      fs_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:fs_liq_cov}','  ',\
      '\\\\')
# FS liq T-Us
print(fs_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(fs_fitTus_liq0.curvetype), ' & ',\
      formata(fs_fitTus_liq0.a),' & ',\
      fs_fitTus_liq0.x_units,', ',fs_fitTus_liq0.f_units,' & ',\
      np.round(100.*fs_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*fs_fitTus_liq0.xrange[1])/100,' & ',\
      int(np.round(fs_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      fs_fitTus_liq0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:fs_liq_T_cov}','  ',\
      '\\\\')

stishovite  &  $U_s(u_p)$ &  1  &  [    1.107070,     1.607013]  &  km/s ,  km/s  &  2.76 $< u_p <$ 3.91  &  34 - 72  &  16  &  Eq. \ref{eq:fs_stv_cov}     \\
stishovite  &  $T(U_s)$ &  2  &  [ 7684.978628, -2052.464734,   226.345708]  &  km/s ,  K  &  5.72 $< U_s <$ 7.62  &  34 - 72  &  21 $^{\S}$  &  Eq. \ref{eq:fs_stv_T_cov}     \\
liquid$^{\ddagger}$  &  $U_s(u_p)$ &  2  &  [    1.230166,     1.681657,    -0.014014]  &  km/s ,  km/s  &  4.76 $< u_p <$ 18.28  &  74 - 1099  &  132  &  Eq. \ref{eq:fs_liq_cov}    \\
liquid$^{\ddagger}$  &  $T(U_s)$ &  3  &  [-1418.437322,   526.816293,   -12.799697,     4.979201]  &  km/s ,  K  &  7.98 $< U_s <$ 21.25  &  74 - 628  &  42 $^{\S}$  &  Eq. \ref{eq:fs_liq_T_cov}    \\


In [14]:
tomatrix(fs_fit_stv0,'_{U_s}','eq:fs_stv_cov')
tomatrix(fs_fitTus_stv0,'_T','eq:fs_stv_T_cov')


% fs_fit_stv_linear 
\label{eq:fs_stv_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.945375e-02} & \textrm{-5.988927e-03}\\
\textrm{-5.988927e-03} & \textrm{1.873065e-03}
\end{bmatrix} \\
% fs_fitTUs_stv_quadratic 
\label{eq:fs_stv_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{9.485799e+07} & \textrm{-2.865272e+07} & \textrm{2.147604e+06}\\
\textrm{-2.865272e+07} & \textrm{8.665182e+06} & \textrm{-6.502616e+05}\\
\textrm{2.147604e+06} & \textrm{-6.502616e+05} & \textrm{4.885815e+04}
\end{bmatrix} \\


In [15]:
tomatrix(fs_fit_liq0,'_{U_s}','eq:fs_liq_cov')
tomatrix(fs_fitTus_liq0,'_T','eq:fs_liq_T_cov')


% fs_fit_liq_quadratic 
\label{eq:fs_liq_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.009273e-02} & \textrm{-1.655528e-03} & \textrm{6.461940e-05}\\
\textrm{-1.655528e-03} & \textrm{2.795143e-04} & \textrm{-1.116380e-05}\\
\textrm{6.461940e-05} & \textrm{-1.116380e-05} & \textrm{4.553390e-07}
\end{bmatrix} \\
% fs_fitTus_liq_cubic 
\label{eq:fs_liq_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{2.107651e+07} & \textrm{-5.238609e+06} & \textrm{4.117767e+05} & \textrm{-1.022102e+04}\\
\textrm{-5.238609e+06} & \textrm{1.309912e+06} & \textrm{-1.036519e+05} & \textrm{2.589192e+03}\\
\textrm{4.117767e+05} & \textrm{-1.036519e+05} & \textrm{8.264540e+03} & \textrm{-2.079991e+02}\\
\textrm{-1.022102e+04} & \textrm{2.589192e+03} & \textrm{-2.079991e+02} & \textrm{5.276461e+00}
\end{bmatrix} \\


### α-Quartz, z-cut

Stishovite range:<br>
z-cut qtz has a multiwave structure below 70 GPa (Wackerle 1962).<br>
Transition from stv region to CaCl₂ region depends on choice of phase boundary. Here using GGA from Tsuchiya et al. 2004.<br>


In [16]:
qtz_singleshock_P = 70 # GPa Wackerle 1962

liqmaxP = np.interp(qtz_fit_liq0.xrange[1],qtz_hug_liq.up,qtz_hug_liq.P)
print(liqmaxP)


stvTmaxP = np.interp(qtz_fitTus_stv0.xrange[1],qtz_hug_stv.us,qtz_hug_stv.P)
stvTminP = np.interp(qtz_fitTus_stv0.xrange[0],qtz_hug_stv.us,qtz_hug_stv.P)
print(stvTminP,stvTmaxP)

liqTmaxP = np.interp(qtz_fitTus_liq0.xrange[1],qtz_hug_liq.us,qtz_hug_liq.P)
liqTminP = np.interp(qtz_fitTus_liq0.xrange[0],qtz_hug_liq.us,qtz_hug_liq.P)
print(liqTminP,liqTmaxP)


stvminP = np.interp(qtz_fit_stv0.xrange[0],qtz_hug_stv.us,qtz_hug_stv.P)
print(stvminP)

stvminP = np.interp(qtz_fit_stv0.xrange[0],qtz_hug_stv.us,qtz_hug_stv.P)
print(stvminP)


1570.6666175697558
57.82270405157359 89.11670538838202
113.85750308553155 530.316854703606
39.53578872527477
39.53578872527477


In [17]:
## Original table
# Region & $P$ (GPa) & Form & $f(x)$ & Coefficients $a_i$ & Fit range (km/s) & $N$ & $\Sigma$ & Notes \\

# qtz stv us-up
print(qtz_fit_stv0.phase_fit, ' & ',\
      int(np.round(stvminP)),'-',int(np.round(qtz_trans_cacl2_P)),' & ',\
      curve2num(qtz_fit_stv0.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(qtz_fit_stv0.a),' & ',\
      np.round(qtz_fit_stv0.xrange[0]*100)/100,'$< u_p <$',np.round(100*qtz_fit_stv0.xrange[1])/100,' & ',\
      qtz_fit_stv0.npts,' & ',\
      'Eq. \\ref{eq:qtz_stv_cov}',' & ',\
      '$\\dagger$ \\\\')
# qtz stv T-us
print(qtz_fitTus_stv0.phase_fit, ' & ',\
      int(np.round(stvTminP)),'-',int(np.round(qtz_trans_cacl2_P)),' & ',\
      curve2num(qtz_fitTus_stv0.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(qtz_fitTus_stv0.a),' & ',\
      np.round(100*qtz_fitTus_stv0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_stv0.xrange[1])/100,' & ',\
      qtz_fitTus_stv0.npts,' & ',\
      'Eq. \\ref{eq:qtz_stv_T_cov}',' & ',\
      '* \\\\')


# qtz cacl2 us-up
print('CaCl$_2$-type', ' & ',\
      int(np.round(qtz_trans_cacl2_P)),'-',int(np.round(qtz_wiggle_P1)),' & ',\
      curve2num(qtz_fit_cacl20.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(qtz_fit_cacl20.a),' & ',\
      np.round(qtz_fit_cacl20.xrange[0]*100)/100,'$< u_p <$',np.round(100*qtz_fit_cacl20.xrange[1])/100,' & ',\
      qtz_fit_cacl20.npts,' & ',\
      'Eq. \\ref{eq:qtz_cacl2_cov}',' & ',\
      '$\\dagger$ \\\\')
# qtz cacl2 T-us
print('CaCl$_2$-type', ' & ',\
      int(np.round(qtz_trans_cacl2_P)),'-',int(np.round(qtz_wiggle_P1)),' & ',\
      curve2num(qtz_fitTus_cacl20.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(qtz_fitTus_cacl20.a),' & ',\
      np.round(100*qtz_fitTus_cacl20.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_cacl20.xrange[1])/100,' & ',\
      qtz_fitTus_cacl20.npts,' & ',\
      'Eq. \\ref{eq:qtz_cacl2_T_cov}',' & ',\
      ' \\\\')


# qtz liq Us-up
print(qtz_fit_liq0.phase_fit, ' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      curve2num(qtz_fit_liq0.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(qtz_fit_liq0.a),' & ',\
      np.round(100*qtz_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*qtz_fit_liq0.xrange[1])/100,' & ',\
      qtz_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:qtz_liq_cov}',' & ',\
      '$\\ddagger$ \\\\')
# qtz liq T-Us
print(qtz_fit_liq0.phase_fit, ' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      curve2num(qtz_fitTus_liq0.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(qtz_fitTus_liq0.a),' & ',\
      np.round(100.*qtz_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_liq0.xrange[1])/100,' & ',\
      qtz_fitTus_liq0.npts,' & ',\
      'Eq. \\ref{eq:qtz_liq_T_cov}',' & ',\
      '$\\ddagger$ \\\\')

stishovite  &  40 - 83  &  1  &  $U_s(u_p)$ &  [    1.525695,     1.776788]  &  2.7 $< u_p <$ 3.52  &  12  &  Eq. \ref{eq:qtz_stv_cov}  &  $\dagger$ \\
stishovite  &  58 - 83  &  2  &  $T(U_s)$ &  [16761.582648, -4576.928231,   375.033976]  &  7.04 $< U_s <$ 8.53  &  5  &  Eq. \ref{eq:qtz_stv_T_cov}  &  * \\
CaCl$_2$-type  &  83 - 108  &  1  &  $U_s(u_p)$ &  [    2.617896,     1.506306]  &  3.77 $< u_p <$ 4.45  &  9  &  Eq. \ref{eq:qtz_cacl2_cov}  &  $\dagger$ \\
CaCl$_2$-type  &  83 - 108  &  1  &  $T(U_s)$ &  [-9101.804182,  1609.024387]  &  8.3 $< U_s <$ 9.32  &  5  &  Eq. \ref{eq:qtz_cacl2_T_cov}  &   \\
liquid  &  114 - 1571  &  3  &  $U_s(u_p)$ &  [    1.756018,     1.849562,    -0.031746,     0.000497]  &  4.57 $< u_p <$ 19.85  &  211  &  Eq. \ref{eq:qtz_liq_cov}  &  $\ddagger$ \\
liquid  &  114 - 530  &  3  &  $T(U_s)$ &  [-4391.275846,  1263.498083,   -81.263343,     5.262475]  &  9.51 $< U_s <$ 18.6  &  2223  &  Eq. \ref{eq:qtz_liq_T_cov}  &  $\ddagger$ \\


In [18]:
## New table columns
# Region & $f(x)$ & Form & Coefficients $a_i$ & $f,x$ units & Fit range & $P$ range (GPa) & $N$ & Cov. Matrix$^{\Sigma}$ \\

# qtz stv us-up
print(qtz_fit_stv0.phase_fit+'$^{\\dagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(qtz_fit_stv0.curvetype), ' & ',\
      formata(qtz_fit_stv0.a),' & ',\
      qtz_fit_stv0.x_units,', ',qtz_fit_stv0.f_units,' & ',\
      np.round(qtz_fit_stv0.xrange[0]*100)/100,'$< u_p <$',np.round(100*qtz_fit_stv0.xrange[1])/100,' & ',\
      int(np.round(stvminP)),'-',int(np.round(qtz_trans_cacl2_P)),' & ',\
      qtz_fit_stv0.npts,' & ',\
      'Eq. \\ref{eq:qtz_stv_cov}','  ',\
      ' \\\\')
# qtz stv T-us
print(qtz_fitTus_stv0.phase_fit+'^*', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(qtz_fitTus_stv0.curvetype), ' & ',\
      formata(qtz_fitTus_stv0.a),' & ',\
      qtz_fitTus_stv0.x_units,', ',qtz_fitTus_stv0.f_units,' & ',\
      np.round(100*qtz_fitTus_stv0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_stv0.xrange[1])/100,' & ',\
      int(np.round(stvTminP)),'-',int(np.round(qtz_trans_cacl2_P)),' & ',\
      qtz_fitTus_stv0.npts,' & ',\
      'Eq. \\ref{eq:qtz_stv_T_cov}','  ',\
      ' \\\\')


# qtz cacl2 us-up
print('CaCl$_2$-type$^{\\dagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(qtz_fit_cacl20.curvetype), ' & ',\
      formata(qtz_fit_cacl20.a),' & ',\
      qtz_fit_cacl20.x_units,', ',qtz_fit_cacl20.f_units,' & ',\
      np.round(qtz_fit_cacl20.xrange[0]*100)/100,'$< u_p <$',np.round(100*qtz_fit_cacl20.xrange[1])/100,' & ',\
      int(np.round(qtz_trans_cacl2_P)),'-',int(np.round(qtz_wiggle_P1)),' & ',\
      qtz_fit_cacl20.npts,' & ',\
      'Eq. \\ref{eq:qtz_cacl2_cov}','  ',\
      '\\\\')
# qtz cacl2 T-us
print('CaCl$_2$-type', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(qtz_fitTus_cacl20.curvetype), ' & ',\
      formata(qtz_fitTus_cacl20.a),' & ',\
      qtz_fitTus_cacl20.x_units,', ',qtz_fitTus_cacl20.f_units,' & ',\
      np.round(100*qtz_fitTus_cacl20.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_cacl20.xrange[1])/100,' & ',\
      int(np.round(qtz_trans_cacl2_P)),'-',int(np.round(qtz_wiggle_P1)),' & ',\
      qtz_fitTus_cacl20.npts,' & ',\
      'Eq. \\ref{eq:qtz_cacl2_T_cov}','  ',\
      ' \\\\')


# qtz liq Us-up
print(qtz_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(qtz_fit_liq0.curvetype), ' & ',\
      formata(qtz_fit_liq0.a),' & ',\
      qtz_fit_liq0.x_units,', ',qtz_fit_liq0.f_units,' & ',\
      np.round(100*qtz_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*qtz_fit_liq0.xrange[1])/100,' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      qtz_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:qtz_liq_cov}','  ',\
      '\\\\')
# qtz liq T-Us
print(qtz_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(qtz_fitTus_liq0.curvetype), ' & ',\
      formata(qtz_fitTus_liq0.a),' & ',\
      qtz_fitTus_liq0.x_units,', ',qtz_fitTus_liq0.f_units,' & ',\
      np.round(100.*qtz_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_liq0.xrange[1])/100,' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      qtz_fitTus_liq0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:qtz_liq_T_cov}','  ',\
      ' \\\\')

stishovite$^{\dagger}$  &  $U_s(u_p)$ &  1  &  [    1.525695,     1.776788]  &  km/s ,  km/s  &  2.7 $< u_p <$ 3.52  &  40 - 83  &  12  &  Eq. \ref{eq:qtz_stv_cov}     \\
stishovite^*  &  $T(U_s)$ &  2  &  [16761.582648, -4576.928231,   375.033976]  &  km/s ,  K  &  7.04 $< U_s <$ 8.53  &  58 - 83  &  5  &  Eq. \ref{eq:qtz_stv_T_cov}     \\
CaCl$_2$-type$^{\dagger}$  &  $U_s(u_p)$ &  1  &  [    2.617896,     1.506306]  &  km/s ,  km/s  &  3.77 $< u_p <$ 4.45  &  83 - 108  &  9  &  Eq. \ref{eq:qtz_cacl2_cov}    \\
CaCl$_2$-type  &  $T(U_s)$ &  1  &  [-9101.804182,  1609.024387]  &  km/s ,  K  &  8.3 $< U_s <$ 9.32  &  83 - 108  &  5  &  Eq. \ref{eq:qtz_cacl2_T_cov}     \\
liquid$^{\ddagger}$  &  $U_s(u_p)$ &  3  &  [    1.756018,     1.849562,    -0.031746,     0.000497]  &  km/s ,  km/s  &  4.57 $< u_p <$ 19.85  &  114 - 1571  &  211  &  Eq. \ref{eq:qtz_liq_cov}    \\
liquid$^{\ddagger}$  &  $T(U_s)$ &  3  &  [-4391.275846,  1263.498083,   -81.263343,     5.262475]  &  km/s ,  K  &  9.

In [19]:
tomatrix(qtz_fit_stv0,'_{U_s}','eq:qtz_stv_cov')
tomatrix(qtz_fitTus_stv0,'_T','eq:qtz_stv_T_cov')



% qtz_fit_stv_linear 
\label{eq:qtz_stv_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{4.325045e-02} & \textrm{-1.350526e-02}\\
\textrm{-1.350526e-02} & \textrm{4.256561e-03}
\end{bmatrix} \\
% qtz_fitTus_stv_quadratic 
\label{eq:qtz_stv_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}\\
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}\\
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}
\end{bmatrix} \\


In [20]:
tomatrix(qtz_fit_cacl20,'_{U_s}','eq:qtz_cacl2_cov')
tomatrix(qtz_fitTus_cacl20,'_T','eq:qtz_cacl2_T_cov')



% qtz_fit_cacl2_linear 
\label{eq:qtz_cacl2_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{9.058877e-03} & \textrm{-2.260451e-03}\\
\textrm{-2.260451e-03} & \textrm{5.672988e-04}
\end{bmatrix} \\
% qtz_fitTus_cacl2_linear 
\label{eq:qtz_cacl2_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{3.685437e+06} & \textrm{-4.218669e+05}\\
\textrm{-4.218669e+05} & \textrm{4.840714e+04}
\end{bmatrix} \\


In [21]:
tomatrix(qtz_fit_liq0,'_{U_s}','eq:qtz_liq_cov')
tomatrix(qtz_fitTus_liq0,'_T','eq:qtz_liq_T_cov')


% qtz_fit_liq_cubic 
\label{eq:qtz_liq_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.611280e-02} & \textrm{-4.719617e-03} & \textrm{4.275493e-04} & \textrm{-1.214917e-05}\\
\textrm{-4.719617e-03} & \textrm{1.423898e-03} & \textrm{-1.318055e-04} & \textrm{3.808307e-06}\\
\textrm{4.275493e-04} & \textrm{-1.318055e-04} & \textrm{1.241910e-05} & \textrm{-3.642407e-07}\\
\textrm{-1.214917e-05} & \textrm{3.808307e-06} & \textrm{-3.642407e-07} & \textrm{1.083072e-08}
\end{bmatrix} \\
% qtz_fitTus_liq_cubic 
\label{eq:qtz_liq_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{8.272339e+06} & \textrm{-1.919116e+06} & \textrm{1.458095e+05} & \textrm{-3.624740e+03}\\
\textrm{-1.919116e+06} & \textrm{4.466991e+05} & \textrm{-3.405111e+04} & \textrm{8.492170e+02}\\
\textrm{1.458095e+05} & \textrm{-3.405111e+04} & \textrm{2.604364e+03} & \textrm{-6.516890e+01}\\
\textrm{-3.624740e+03} & \textrm{8.492170e+02} & \textrm{-6.516890e+01} & \textrm{1.636324e+00}
\end{bmatrix} \\


## Quartz new 2 regions

In [22]:
## New table columns 2 regions
# Region & $f(x)$ & Form & Coefficients $a_i$ & $f,x$ units & Fit range & $P$ range (GPa) & $N$ & Cov. Matrix$^{\Sigma}$ \\

# qtz solid us-up
print(qtz_fit_solid0.phase_fit+'$^{\\dagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(qtz_fit_solid0.curvetype), ' & ',\
      formata(qtz_fit_solid0.a),' & ',\
      qtz_fit_solid0.x_units,', ',qtz_fit_solid0.f_units,' & ',\
      np.round(qtz_fit_solid0.xrange[0]*100)/100,'$< u_p <$',np.round(100*qtz_fit_solid0.xrange[1])/100,' & ',\
      int(np.round(qtz_singleshock_P)),'-',int(np.round(qtz_solidus_P)),' & ',\
      qtz_fit_solid0.npts,' & ',\
      'Eq. \\ref{eq:qtz_solid_cov}','  ',\
      ' \\\\')
# qtz solid T-us
print(qtz_fitTus_solid0.phase_fit+'^*', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(qtz_fitTus_solid0.curvetype), ' & ',\
      formata(qtz_fitTus_solid0.a),' & ',\
      qtz_fitTus_solid0.x_units,', ',qtz_fitTus_solid0.f_units,' & ',\
      np.round(100*qtz_fitTus_solid0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_solid0.xrange[1])/100,' & ',\
      int(np.round(qtz_singleshock_P)),'-',int(np.round(qtz_solidus_P)),' & ',\
      qtz_fitTus_solid0.npts,' & ',\
      'Eq. \\ref{eq:qtz_solid_T_cov}','  ',\
      ' \\\\')


# qtz liq Us-up
print(qtz_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(qtz_fit_liq0.curvetype), ' & ',\
      formata(qtz_fit_liq0.a),' & ',\
      qtz_fit_liq0.x_units,', ',qtz_fit_liq0.f_units,' & ',\
      np.round(100*qtz_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*qtz_fit_liq0.xrange[1])/100,' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      qtz_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:qtz_liq_cov}','  ',\
      '\\\\')
# qtz liq T-Us
print(qtz_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(qtz_fitTus_liq0.curvetype), ' & ',\
      formata(qtz_fitTus_liq0.a),' & ',\
      qtz_fitTus_liq0.x_units,', ',qtz_fitTus_liq0.f_units,' & ',\
      np.round(100.*qtz_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*qtz_fitTus_liq0.xrange[1])/100,' & ',\
      int(np.round(qtz_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      qtz_fitTus_liq0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:qtz_liq_T_cov}','  ',\
      ' \\\\')

solid$^{\dagger}$  &  $U_s(u_p)$ &  1  &  [    1.969055,     1.665077]  &  km/s ,  km/s  &  2.7 $< u_p <$ 4.45  &  70 - 101  &  19  &  Eq. \ref{eq:qtz_solid_cov}     \\
solid^*  &  $T(U_s)$ &  2  &  [13817.143622, -3554.253516,   289.960802]  &  km/s ,  K  &  7.42 $< U_s <$ 9.32  &  70 - 101  &  6  &  Eq. \ref{eq:qtz_solid_T_cov}     \\
liquid$^{\ddagger}$  &  $U_s(u_p)$ &  3  &  [    1.756018,     1.849562,    -0.031746,     0.000497]  &  km/s ,  km/s  &  4.57 $< u_p <$ 19.85  &  114 - 1571  &  211  &  Eq. \ref{eq:qtz_liq_cov}    \\
liquid$^{\ddagger}$  &  $T(U_s)$ &  3  &  [-4391.275846,  1263.498083,   -81.263343,     5.262475]  &  km/s ,  K  &  9.51 $< U_s <$ 18.6  &  114 - 530  &  2223 $^{\S}$  &  Eq. \ref{eq:qtz_liq_T_cov}     \\


In [23]:
tomatrix(qtz_fit_solid0,'_{U_s}','eq:qtz_solid_cov')
tomatrix(qtz_fitTus_solid0,'_T','eq:qtz_solid_T_cov')



% qtz_fit_solid_linear 
\label{eq:qtz_solid_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{4.430110e-03} & \textrm{-1.126518e-03}\\
\textrm{-1.126518e-03} & \textrm{2.898086e-04}
\end{bmatrix} \\
% qtz_fitTus_solid_quadratic 
\label{eq:qtz_solid_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{1.562536e+08} & \textrm{-3.818662e+07} & \textrm{2.315155e+06}\\
\textrm{-3.818662e+07} & \textrm{9.341511e+06} & \textrm{-5.669128e+05}\\
\textrm{2.315155e+06} & \textrm{-5.669128e+05} & \textrm{3.444002e+04}
\end{bmatrix} \\


In [24]:
tomatrix(qtz_fit_liq0,'_{U_s}','eq:qtz_liq_cov')
tomatrix(qtz_fitTus_liq0,'_T','eq:qtz_liq_T_cov')


% qtz_fit_liq_cubic 
\label{eq:qtz_liq_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.611280e-02} & \textrm{-4.719617e-03} & \textrm{4.275493e-04} & \textrm{-1.214917e-05}\\
\textrm{-4.719617e-03} & \textrm{1.423898e-03} & \textrm{-1.318055e-04} & \textrm{3.808307e-06}\\
\textrm{4.275493e-04} & \textrm{-1.318055e-04} & \textrm{1.241910e-05} & \textrm{-3.642407e-07}\\
\textrm{-1.214917e-05} & \textrm{3.808307e-06} & \textrm{-3.642407e-07} & \textrm{1.083072e-08}
\end{bmatrix} \\
% qtz_fitTus_liq_cubic 
\label{eq:qtz_liq_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{8.272339e+06} & \textrm{-1.919116e+06} & \textrm{1.458095e+05} & \textrm{-3.624740e+03}\\
\textrm{-1.919116e+06} & \textrm{4.466991e+05} & \textrm{-3.405111e+04} & \textrm{8.492170e+02}\\
\textrm{1.458095e+05} & \textrm{-3.405111e+04} & \textrm{2.604364e+03} & \textrm{-6.516890e+01}\\
\textrm{-3.624740e+03} & \textrm{8.492170e+02} & \textrm{-6.516890e+01} & \textrm{1.636324e+00}
\end{bmatrix} \\


### Coesite



In [25]:

liqmaxP = np.interp(cst_fit_liq0.xrange[1],cst_hug_liq.up,cst_hug_liq.P)
print(liqmaxP)


stvTmaxP = np.interp(cst_fitTus_cacl20.xrange[1],cst_hug_cacl2.us,cst_hug_cacl2.P)
stvTminP = np.interp(cst_fitTus_cacl20.xrange[0],cst_hug_cacl2.us,cst_hug_cacl2.P)
print(stvTminP,stvTmaxP)

liqTmaxP = np.interp(cst_fitTus_liq0.xrange[1],cst_hug_liq.us,cst_hug_liq.P)
liqTminP = np.interp(cst_fitTus_liq0.xrange[0],cst_hug_liq.us,cst_hug_liq.P)
print(liqTminP,liqTmaxP)


stvminP = np.interp(cst_fit_cacl20.xrange[0],cst_hug_cacl2.us,cst_hug_cacl2.P)
print(stvminP)

stvminP = np.interp(cst_fit_cacl20.xrange[0],cst_hug_cacl2.us,cst_hug_cacl2.P)
print(stvminP)



954.0814030701375
91.39509724024062 125.09503811220782
135.19934814945572 525.2842041049234
49.615945694694375
49.615945694694375


In [26]:
## Original table
# Region & $P$ (GPa) & Form & $f(x)$ & Coefficients $a_i$ & Fit range (km/s) & $N$ & $\Sigma$ & Notes \\

# cst stv us-up
print('Solid', ' & ',\
      int(np.round(stvminP)),'-',int(np.round(cst_sol_cross.P)),' & ',\
      curve2num(cst_fit_cacl20.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(cst_fit_cacl20.a),' & ',\
      np.round(cst_fit_cacl20.xrange[0]*100)/100,'$< u_p <$',np.round(100*cst_fit_cacl20.xrange[1])/100,' & ',\
      cst_fit_cacl20.npts,' & ',\
      'Eq. \\ref{eq:cst_solid_cov}',' & ',\
      '$\\dagger$ \\\\')
# cst stv T-us
print('Solid', ' & ',\
      int(np.round(stvTminP)),'-',int(np.round(cst_sol_cross.P)),' & ',\
      curve2num(cst_fitTus_cacl20.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(cst_fitTus_cacl20.a),' & ',\
      np.round(100*cst_fitTus_cacl20.xrange[0])/100,'$< U_s <$',np.round(100*cst_fitTus_cacl20.xrange[1])/100,' & ',\
      cst_fitTus_cacl20.npts,' & ',\
      'Eq. \\ref{eq:cst_solid_T_cov}',' & ',\
      '* \\\\')
# cst liq Us-up
print(cst_fit_liq0.phase_fit, ' & ',\
      int(np.round(cst_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      curve2num(cst_fit_liq0.curvetype), ' & ',\
      '$U_s(u_p)$ & ',\
      formata(cst_fit_liq0.a),' & ',\
      np.round(100*cst_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*cst_fit_liq0.xrange[1])/100,' & ',\
      cst_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:cst_liq_cov}',' & ',\
      '$\\ddagger$ \\\\')
# cst liq T-Us
print(cst_fit_liq0.phase_fit, ' & ',\
      int(np.round(cst_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      curve2num(cst_fitTus_liq0.curvetype), ' & ',\
      '$T(U_s)$ & ',\
      formata(cst_fitTus_liq0.a),' & ',\
      np.round(100.*cst_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*cst_fitTus_liq0.xrange[1])/100,' & ',\
      cst_fitTus_liq0.npts,' & ',\
      'Eq. \\ref{eq:cst_liq_T_cov}',' & ',\
      '$\\ddagger$ \\\\')

Solid  &  50 - 126  &  1  &  $U_s(u_p)$ &  [    2.595321,     1.680553]  &  2.86 $< u_p <$ 4.05  &  6  &  Eq. \ref{eq:cst_solid_cov}  &  $\dagger$ \\
Solid  &  91 - 126  &  2  &  $T(U_s)$ &  [28300.890211, -6631.873439,   441.706278]  &  8.67 $< U_s <$ 9.88  &  5  &  Eq. \ref{eq:cst_solid_T_cov}  &  * \\
liquid  &  135 - 954  &  1  &  $U_s(u_p)$ &  [    4.005630,     1.407401]  &  6.57 $< u_p <$ 13.88  &  6  &  Eq. \ref{eq:cst_liq_cov}  &  $\ddagger$ \\
liquid  &  135 - 525  &  2  &  $T(U_s)$ &  [-1474.221167,  -116.724248,    68.713544]  &  10.32 $< U_s <$ 18.04  &  834  &  Eq. \ref{eq:cst_liq_T_cov}  &  $\ddagger$ \\


In [27]:
## New table columns
# Region & $f(x)$ & Form & Coefficients $a_i$ & $f,x$ units & Fit range & $P$ range (GPa) & $N$ & Cov. Matrix$^{\Sigma}$ \\

# cst stv us-up
print('solid$^{\\dagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(cst_fit_cacl20.curvetype), ' & ',\
      formata(cst_fit_cacl20.a),' & ',\
      cst_fit_cacl20.x_units,', ',cst_fit_cacl20.f_units,' & ',\
      np.round(cst_fit_cacl20.xrange[0]*100)/100,'$< u_p <$',np.round(100*cst_fit_cacl20.xrange[1])/100,' & ',\
      int(np.round(stvminP)),'-',int(np.round(cst_sol_cross.P)),' & ',\
      cst_fit_cacl20.npts,' & ',\
      'Eq. \\ref{eq:cst_solid_cov}','  ',\
      ' \\\\')
# cst stv T-us
print('solid$^*$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(cst_fitTus_cacl20.curvetype), ' & ',\
      formata(cst_fitTus_cacl20.a),' & ',\
      cst_fitTus_cacl20.x_units,', ',cst_fitTus_cacl20.f_units,' & ',\
      np.round(100*cst_fitTus_cacl20.xrange[0])/100,'$< U_s <$',np.round(100*cst_fitTus_cacl20.xrange[1])/100,' & ',\
      int(np.round(stvTminP)),'-',int(np.round(cst_sol_cross.P)),' & ',\
      cst_fitTus_cacl20.npts,' & ',\
      'Eq. \\ref{eq:cst_solid_T_cov}','  ',\
      ' \\\\')
# cst liq Us-up
print(cst_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(cst_fit_liq0.curvetype), ' & ',\
      formata(cst_fit_liq0.a),' & ',\
      cst_fit_liq0.x_units,', ',cst_fit_liq0.f_units,' & ',\
      np.round(100*cst_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*cst_fit_liq0.xrange[1])/100,' & ',\
      int(np.round(cst_wiggle_P2)),'-',int(np.round(liqmaxP)),' & ',\
      cst_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:cst_liq_cov}','  ',\
      ' \\\\')
# cst liq T-Us
print(cst_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(cst_fitTus_liq0.curvetype), ' & ',\
      formata(cst_fitTus_liq0.a),' & ',\
      cst_fitTus_liq0.x_units,', ',cst_fitTus_liq0.f_units,' & ',\
      np.round(100.*cst_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*cst_fitTus_liq0.xrange[1])/100,' & ',\
      int(np.round(cst_wiggle_P2)),'-',int(np.round(liqTmaxP)),' & ',\
      cst_fitTus_liq0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:cst_liq_T_cov}','  ',\
      ' \\\\')

solid$^{\dagger}$  &  $U_s(u_p)$ &  1  &  [    2.595321,     1.680553]  &  km/s ,  km/s  &  2.86 $< u_p <$ 4.05  &  50 - 126  &  6  &  Eq. \ref{eq:cst_solid_cov}     \\
solid$^*$  &  $T(U_s)$ &  2  &  [28300.890211, -6631.873439,   441.706278]  &  km/s ,  K  &  8.67 $< U_s <$ 9.88  &  91 - 126  &  5  &  Eq. \ref{eq:cst_solid_T_cov}     \\
liquid$^{\ddagger}$  &  $U_s(u_p)$ &  1  &  [    4.005630,     1.407401]  &  km/s ,  km/s  &  6.57 $< u_p <$ 13.88  &  135 - 954  &  6  &  Eq. \ref{eq:cst_liq_cov}     \\
liquid$^{\ddagger}$  &  $T(U_s)$ &  2  &  [-1474.221167,  -116.724248,    68.713544]  &  km/s ,  K  &  10.32 $< U_s <$ 18.04  &  135 - 525  &  834 $^{\S}$  &  Eq. \ref{eq:cst_liq_T_cov}     \\


In [28]:
tomatrix(cst_fit_cacl20,'_{U_s}','eq:cst_solid_cov')
tomatrix(cst_fitTus_cacl20,'_T','eq:cst_solid_T_cov')


% cst_fit_cacl2_linear 
\label{eq:cst_solid_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.550646e-01} & \textrm{-4.520090e-02}\\
\textrm{-4.520090e-02} & \textrm{1.334589e-02}
\end{bmatrix} \\
% cst_fitTus_cacl2_quadratic 
\label{eq:cst_solid_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}\\
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}\\
\textrm{0.000000e+00} & \textrm{0.000000e+00} & \textrm{0.000000e+00}
\end{bmatrix} \\


In [29]:
tomatrix(cst_fit_liq0,'_{U_s}','eq:cst_liq_cov')
tomatrix(cst_fitTus_liq0,'_T','eq:cst_liq_T_cov')


% cst_fit_liq_linear 
\label{eq:cst_liq_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{3.209859e-01} & \textrm{-3.182132e-02}\\
\textrm{-3.182132e-02} & \textrm{3.389221e-03}
\end{bmatrix} \\
% cst_fitTus_liq_quadratic 
\label{eq:cst_liq_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{1.031770e+06} & \textrm{-1.631596e+05} & \textrm{6.331791e+03}\\
\textrm{-1.631596e+05} & \textrm{2.591774e+04} & \textrm{-1.010834e+03}\\
\textrm{6.331791e+03} & \textrm{-1.010834e+03} & \textrm{3.965336e+01}
\end{bmatrix} \\


### Stishovite liquid region only



In [30]:

liqminP = np.interp(stv_fit_liq0.xrange[0],stv_hug_liq.up,stv_hug_liq.P)
liqmaxP = np.interp(stv_fit_liq0.xrange[1],stv_hug_liq.up,stv_hug_liq.P)
print(liqminP,liqmaxP)


liqTmaxP = np.interp(stv_fitTus_liq0.xrange[1],stv_hug_liq.us,stv_hug_liq.P)
liqTminP = np.interp(stv_fitTus_liq0.xrange[0],stv_hug_liq.us,stv_hug_liq.P)
print(liqTminP,liqTmaxP)



1034.4080105307196 2250.092157669525
590.2443571034056 2516.4994508249974


In [31]:
## New table columns
# Region & $f(x)$ & Form & Coefficients $a_i$ & $f,x$ units & Fit range & $P$ range (GPa) & $N$ & Cov. Matrix$^{\Sigma}$ \\


# stv liq Us-up
print(stv_fit_liq0.phase_fit, ' & ',\
      '$U_s(u_p)$ & ',\
      curve2num(stv_fit_liq0.curvetype), ' & ',\
      formata(stv_fit_liq0.a),' & ',\
      stv_fit_liq0.x_units,', ',stv_fit_liq0.f_units,' & ',\
      np.round(100*stv_fit_liq0.xrange[0])/100,'$< u_p <$',np.round(100*stv_fit_liq0.xrange[1])/100,' & ',\
      int(np.round(liqminP)),'-',int(np.round(liqmaxP)),' & ',\
      stv_fit_liq0.npts,' & ',\
      'Eq. \\ref{eq:stv_liq_cov}','  ',\
      ' \\\\')
# stv liq T-Us
print(stv_fit_liq0.phase_fit+'$^{\\ddagger}$', ' & ',\
      '$T(U_s)$ & ',\
      curve2num(stv_fitTus_liq0.curvetype), ' & ',\
      formata(stv_fitTus_liq0.a),' & ',\
      stv_fitTus_liq0.x_units,', ',stv_fitTus_liq0.f_units,' & ',\
      np.round(100.*stv_fitTus_liq0.xrange[0])/100,'$< U_s <$',np.round(100*stv_fitTus_liq0.xrange[1])/100,' & ',\
      int(np.round(stv_wiggle2.P)),'-',int(np.round(liqTmaxP)),' & ',\
      stv_fitTus_liq0.npts,'$^{\\S}$',' & ',\
      'Eq. \\ref{eq:stv_liq_T_cov}','  ',\
      ' \\\\')

liquid  &  $U_s(u_p)$ &  1  &  [    7.133055,     1.308495]  &  km/s ,  km/s  &  11.12 $< u_p <$ 17.48  &  1034 - 2250  &  6  &  Eq. \ref{eq:stv_liq_cov}     \\
liquid$^{\ddagger}$  &  $T(U_s)$ &  4  &  [-5.441655e+05, 9.657878e+04, -6.378602e+03, 1.870840e+02, -1.989250e+00]  &  km/s ,  K  &  17.45 $< U_s <$ 31.5  &  586 - 2516  &  282 $^{\S}$  &  Eq. \ref{eq:stv_liq_T_cov}     \\


In [32]:
tomatrix(stv_fit_liq0,'_{U_s}','eq:stv_liq_cov')
tomatrix(stv_fitTus_liq0,'_T','eq:stv_liq_T_cov')


% stv_fit_liq_linear_up18 
\label{eq:stv_liq_cov} 
\Sigma_{U_s} &=& \begin{bmatrix}
\textrm{1.304334e+00} & \textrm{-9.247795e-02}\\
\textrm{-9.247795e-02} & \textrm{6.760612e-03}
\end{bmatrix} \\
% stv_fitTus_liq_fourthpoly 
\label{eq:stv_liq_T_cov} 
\Sigma_T &=& \begin{bmatrix}
\textrm{1.550367e+10} & \textrm{-2.707326e+09} & \textrm{1.751251e+08} & \textrm{-4.972154e+06} & \textrm{5.227816e+04}\\
\textrm{-2.707326e+09} & \textrm{4.734080e+08} & \textrm{-3.066469e+07} & \textrm{8.718216e+05} & \textrm{-9.178773e+03}\\
\textrm{1.751251e+08} & \textrm{-3.066469e+07} & \textrm{1.989057e+06} & \textrm{-5.662966e+04} & \textrm{5.970362e+02}\\
\textrm{-4.972154e+06} & \textrm{8.718216e+05} & \textrm{-5.662966e+04} & \textrm{1.614562e+03} & \textrm{-1.704597e+01}\\
\textrm{5.227816e+04} & \textrm{-9.178773e+03} & \textrm{5.970362e+02} & \textrm{-1.704597e+01} & \textrm{1.802187e-01}
\end{bmatrix} \\


## Tables 3 and D1: Intersection range with melt curve

In [33]:

fs_T1 = int(np.round(np.interp(fs_solidus_P,stsmcsmooth.P,stsmcsmooth.T)))
fs_T2 = int(np.round(np.interp(fs_liquidus_P,stsmcsmooth.P,stsmcsmooth.T)))
fs_P1 = int(fs_solidus_P)
fs_P2 = int(fs_liquidus_P)

qtz_T1 = int(np.round(np.interp(qtz_solidus_P,stsmcsmooth.P,stsmcsmooth.T)))
qtz_T2 = int(np.round(np.interp(qtz_liquidus_P,stsmcsmooth.P,stsmcsmooth.T)))
qtz_P1 = int(qtz_solidus_P)
qtz_P2 = int(qtz_liquidus_P)

cst_T1 = int(np.round(np.interp(cst_solidus_P,stsmcsmooth.P,stsmcsmooth.T)))
cst_T2 = int(np.round(np.interp(cst_liquidus_P,stsmcsmooth.P,stsmcsmooth.T)))
cst_P1 = int(cst_solidus_P)
cst_P2 = int(cst_liquidus_P)

stv_T1 = int(np.round(np.interp(stv_solidus_P,stsmcsmooth.P,stsmcsmooth.T)))
stv_T2 = int(np.round(np.interp(stv_liquidus_P,stsmcsmooth.P,stsmcsmooth.T)))
stv_P1 = int(stv_solidus_P)
stv_P2 = int(stv_liquidus_P)


In [34]:

print('Fused silica & ',fs_P1,', ',fs_T1,' & ',fs_P2,', ',fs_T2,' \\\\')
print('$\\alpha$-Quartz (z-cut) & ',qtz_P1,', ',qtz_T1,' & ',qtz_P2,', ',qtz_T2,' \\\\')
print('Coesite & ',cst_P1,', ',cst_T1,' & ',cst_P2,', ',cst_T2,' \\\\')
print('Stishovite & ',stv_P1,', ',stv_T1,' & ',stv_P2,', ',stv_T2,' \\\\')


Fused silica &  72 ,  5329  &  90 ,  5598  \\
$\alpha$-Quartz (z-cut) &  100 ,  5695  &  145 ,  6117  \\
Coesite &  125 ,  5932  &  183 ,  6449  \\
Stishovite &  490 ,  8705  &  632 ,  9623  \\


In [35]:
def fm1(x):
    return int(np.round(x*10))/10
def fm2(x):
    return int(np.round(x*100))/100
def fmi(x):
    return int(np.round(x))


In [36]:
# with error bars
fs_pt_sol = intersection(fs_hug_stv.P,fs_hug_stv.T,stsmcsmooth.P,stsmcsmooth.T)
fs_pt_sol_ustd = intersection(fs_hug_stv.P,fs_hug_stv.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
fs_pt_sol_lstd = intersection(fs_hug_stv.P,fs_hug_stv.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
fs_rho_sol = np.interp(fs_pt_sol[0],fs_hug_stv.P,fs_hug_stv.rho)[0]
fs_rho_sol_ustd = np.interp(fs_pt_sol_ustd[0],fs_hug_stv.P,fs_hug_stv.rho)[0]
fs_rho_sol_lstd = np.interp(fs_pt_sol_lstd[0],fs_hug_stv.P,fs_hug_stv.rho)[0]

fs_pt_liq = intersection(fs_hug_liq.P,fs_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T)
fs_pt_liq_ustd = intersection(fs_hug_liq.P,fs_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
fs_pt_liq_lstd = intersection(fs_hug_liq.P,fs_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
fs_rho_liq = np.interp(fs_pt_liq[0],fs_hug_liq.P,fs_hug_liq.rho)[0]
fs_rho_liq_ustd = np.interp(fs_pt_liq_ustd[0],fs_hug_liq.P,fs_hug_liq.rho)[0]
fs_rho_liq_lstd = np.interp(fs_pt_liq_lstd[0],fs_hug_liq.P,fs_hug_liq.rho)[0]

print(fs_pt_sol)
print(fs_rho_sol)
print(fs_pt_sol_ustd)
print(fs_rho_sol_ustd)
print(fs_pt_sol_lstd)
print(fs_rho_sol_lstd)


(array([69.20468911]), array([5270.40997026]))
4.71499601443722
(array([73.70425101]), array([5604.64710124]))
4.741083056675837
(array([64.32306043]), array([4923.01541087]))
4.68419899731125


In [48]:
print('Fused silica & ',fm1(fs_pt_sol[0][0]),' & ',fmi(fs_pt_sol[1][0]) ,' & ',fm2(fs_rho_sol),' & ',fm1(fs_pt_liq[0][0]),' & ',fmi(fs_pt_liq[1][0]),' & ', fm2(fs_rho_liq),' \\\\')
print('Fused silica (+ε) & ',fm1(fs_pt_sol_ustd[0][0]),' & ',fmi(fs_pt_sol_ustd[1][0]),' & ', fm2(fs_rho_sol_ustd),' & ',fm1(fs_pt_liq_ustd[0][0]),' & ',fmi(fs_pt_liq_ustd[1][0]),' & ', fm2(fs_rho_liq_ustd),' \\\\')
print('Fused silica (-ε) & ',fm1(fs_pt_sol_lstd[0][0]),' & ',fmi(fs_pt_sol_lstd[1][0]),' & ', fm2(fs_rho_sol_lstd),' & ',fm1(fs_pt_liq_lstd[0][0]),' & ',fmi(fs_pt_liq_lstd[1][0]),' & ', fm2(fs_rho_liq_lstd),' \\\\')


Fused silica &  69.2  &  5270  &  4.71  &  90.5  &  5598  &  4.71  \\
Fused silica (+ε) &  73.7  &  5605  &  4.74  &  95.1  &  5893  &  4.74  \\
Fused silica (-ε) &  64.3  &  4923  &  4.68  &  85.9  &  5302  &  4.68  \\


In [38]:
# with error bars
qtz_pt_sol = intersection(qtz_hug_solid.P,qtz_hug_solid.T,stsmcsmooth.P,stsmcsmooth.T)
qtz_pt_sol_ustd = intersection(qtz_hug_solid.P,qtz_hug_solid.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
qtz_pt_sol_lstd = intersection(qtz_hug_solid.P,qtz_hug_solid.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
qtz_rho_sol = np.interp(qtz_pt_sol[0],qtz_hug_solid.P,qtz_hug_solid.rho)[0]
qtz_rho_sol_ustd = np.interp(qtz_pt_sol_ustd[0],qtz_hug_solid.P,qtz_hug_solid.rho)[0]
qtz_rho_sol_lstd = np.interp(qtz_pt_sol_lstd[0],qtz_hug_solid.P,qtz_hug_solid.rho)[0]

qtz_pt_liq = intersection(qtz_hug_liq.P,qtz_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T)
qtz_pt_liq_ustd = intersection(qtz_hug_liq.P,qtz_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
qtz_pt_liq_lstd = intersection(qtz_hug_liq.P,qtz_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
qtz_rho_liq = np.interp(qtz_pt_liq[0],qtz_hug_liq.P,qtz_hug_liq.rho)[0]
qtz_rho_liq_ustd = np.interp(qtz_pt_liq_ustd[0],qtz_hug_liq.P,qtz_hug_liq.rho)[0]
qtz_rho_liq_lstd = np.interp(qtz_pt_liq_lstd[0],qtz_hug_liq.P,qtz_hug_liq.rho)[0]

print(qtz_pt_sol)
print(qtz_rho_sol)
print(qtz_pt_sol_ustd)
print(qtz_rho_sol_ustd)
print(qtz_pt_sol_lstd)
print(qtz_rho_sol_lstd)


(array([107.34170175]), array([5760.73944828]))
5.026647127956913
(array([111.47870256]), array([6050.06009516]))
5.046877253167954
(array([103.04396282]), array([5469.62015319]))
5.004649728775451


In [50]:

print('α-Quartz (z-cut) & ',fm1(qtz_pt_sol[0][0]),' & ',fmi(qtz_pt_sol[1][0]) ,' & ',fm2(qtz_rho_sol),' & ',fm1(qtz_pt_liq[0][0]),' & ',fmi(qtz_pt_liq[1][0]),' & ', fm2(qtz_rho_liq),' \\\\')
print('α-Quartz (z-cut) (+ε) & ',fm1(qtz_pt_sol_ustd[0][0]),' & ',fmi(qtz_pt_sol_ustd[1][0]),' & ', fm2(qtz_rho_sol_ustd),' & ',fm1(qtz_pt_liq_ustd[0][0]),' & ',fmi(qtz_pt_liq_ustd[1][0]),' & ', fm2(qtz_rho_liq_ustd),' \\\\')
print('α-Quartz (z-cut) (-ε) & ',fm1(qtz_pt_sol_lstd[0][0]),' & ',fmi(qtz_pt_sol_lstd[1][0]),' & ', fm2(qtz_rho_sol_lstd),' & ',fm1(qtz_pt_liq_lstd[0][0]),' & ',fmi(qtz_pt_liq_lstd[1][0]),' & ', fm2(qtz_rho_liq_lstd),' \\\\')


α-Quartz (z-cut) &  107.3  &  5761  &  5.03  &  145.8  &  6117  &  5.21  \\
α-Quartz (z-cut) (+ε) &  111.5  &  6050  &  5.05  &  153.5  &  6436  &  5.24  \\
α-Quartz (z-cut) (-ε) &  103.0  &  5470  &  5.0  &  138.0  &  5797  &  5.17  \\


In [41]:
# with error bars
cst_pt_sol = intersection(cst_hug_cacl2.P,cst_hug_cacl2.T,stsmcsmooth.P,stsmcsmooth.T)
cst_pt_sol_ustd = intersection(cst_hug_cacl2.P,cst_hug_cacl2.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
cst_pt_sol_lstd = intersection(cst_hug_cacl2.P,cst_hug_cacl2.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
cst_rho_sol = np.interp(cst_pt_sol[0],cst_hug_cacl2.P,cst_hug_cacl2.rho)[0]
cst_rho_sol_ustd = np.interp(cst_pt_sol_ustd[0],cst_hug_cacl2.P,cst_hug_cacl2.rho)[0]
cst_rho_sol_lstd = np.interp(cst_pt_sol_lstd[0],cst_hug_cacl2.P,cst_hug_cacl2.rho)[0]

cst_pt_liq = intersection(cst_hug_liq.P,cst_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T)
cst_pt_liq_ustd = intersection(cst_hug_liq.P,cst_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
cst_pt_liq_lstd = intersection(cst_hug_liq.P,cst_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
cst_rho_liq = np.interp(cst_pt_liq[0],cst_hug_liq.P,cst_hug_liq.rho)[0]
cst_rho_liq_ustd = np.interp(cst_pt_liq_ustd[0],cst_hug_liq.P,cst_hug_liq.rho)[0]
cst_rho_liq_lstd = np.interp(cst_pt_liq_lstd[0],cst_hug_liq.P,cst_hug_liq.rho)[0]



In [51]:

print('Coesite (z-cut) & ',fm1(cst_pt_sol[0][0]),' & ',fmi(cst_pt_sol[1][0]) ,' & ',fm2(cst_rho_sol),' & ',fm1(cst_pt_liq[0][0]),' & ',fmi(cst_pt_liq[1][0]),' & ', fm2(cst_rho_liq),' \\\\')
print('Coesite (z-cut) (+ε) & ',fm1(cst_pt_sol_ustd[0][0]),' & ',fmi(cst_pt_sol_ustd[1][0]),' & ', fm2(cst_rho_sol_ustd),' & ',fm1(cst_pt_liq_ustd[0][0]),' & ',fmi(cst_pt_liq_ustd[1][0]),' & ', fm2(cst_rho_liq_ustd),' \\\\')
print('Coesite (z-cut) (-ε) & ',fm1(cst_pt_sol_lstd[0][0]),' & ',fmi(cst_pt_sol_lstd[1][0]),' & ', fm2(cst_rho_sol_lstd),' & ',fm1(cst_pt_liq_lstd[0][0]),' & ',fmi(cst_pt_liq_lstd[1][0]),' & ', fm2(cst_rho_liq_lstd),' \\\\')


Coesite (z-cut) &  125.6  &  5932  &  5.21  &  183.6  &  6449  &  5.46  \\
Coesite (z-cut) (+ε) &  129.6  &  6219  &  5.22  &  192.4  &  6774  &  5.51  \\
Coesite (z-cut) (-ε) &  121.4  &  5643  &  5.18  &  174.8  &  6124  &  5.42  \\


In [58]:
# with error bars

stv_pt_liq = intersection(stv_hug_liq.P,stv_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T)
stv_pt_liq_ustd = intersection(stv_hug_liq.P,stv_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T+stsmcsmooth.T_std)
stv_pt_liq_lstd = intersection(stv_hug_liq.P,stv_hug_liq.T,stsmcsmooth.P,stsmcsmooth.T-stsmcsmooth.T_std)
stv_rho_liq = np.interp(stv_pt_liq[0],stv_hug_liq.P,stv_hug_liq.rho)[0]
stv_rho_liq_ustd = np.interp(stv_pt_liq_ustd[0],stv_hug_liq.P,stv_hug_liq.rho)[0]
stv_rho_liq_lstd = np.interp(stv_pt_liq_lstd[0],stv_hug_liq.P,stv_hug_liq.rho)[0]



In [59]:
print('Stishovite      & & & & ',fm1(stv_pt_liq[0][0]),' & ',fmi(stv_pt_liq[1][0]) ,' & ',fm2(stv_rho_liq),' \\\\')
print('Stishovite (+ε) & & & & ',fm1(stv_pt_liq_ustd[0][0]),' & ',fmi(stv_pt_liq_ustd[1][0]),' & ', fm2(stv_rho_liq_ustd),' \\\\')
print('Stishovite (-ε) & & & & ',fm1(stv_pt_liq_lstd[0][0]),' & ',fmi(stv_pt_liq_lstd[1][0]),' & ', fm2(stv_rho_liq_lstd),' \\\\')


Stishovite      & & & &  632.9  &  9623  &  7.94  \\
Stishovite (+ε) & & & &  645.8  &  9949  &  7.98  \\
Stishovite (-ε) & & & &  620.5  &  9298  &  7.91  \\
